# IS 362: Final Project Part 1 -- Data Preparation
### Neighborhood Class, Transit Access, and COVID-19 Outcomes in New York City

**Student:** Daniel Foulen  

**Class:** IS 362  

**Date:** 5/10/2026

## Background

New York City's subway system did not shut down during COVID-19, but ridership 
collapsed. As the city recovered, that recovery was not even. Some neighborhoods 
bounced back. Others did not.

This notebook builds the dataset to examine why. It pulls subway ridership data, 
COVID-19 outcomes, and socioeconomic indicators from three public sources, joins 
them to a common neighborhood geography, and produces a single analysis-ready file.

---

## Applications

The prepared dataset supports the analysis in Part 2, where ridership 
recovery patterns are examined across neighborhood income tiers. 

The final output, `merged_modzcta.geojson`, combines geographic boundaries, 
transit ridership, COVID-19 outcomes, and socioeconomic indicators into a 
single file ready for analysis and visualization.

---

## Data Sources and Configuration

Three public sources are used:

- MTA Subway Hourly Ridership via the NY Open Data Socrata API
- COVID-19 cumulative outcomes by neighborhood via the NYC Department of Health
- Median income and poverty estimates via the U.S. Census Bureau ACS 2021

All URLs, file paths, and credentials are defined in this step. Every file 
produced by this notebook is cached to /data/. Rerunning after the first 
successful execution loads from disk and requires no internet connection.

---

## Step 1: Imports and Configuration

The following libraries are required. If running in a fresh environment, 
install them before executing this notebook:

```bash
pip install requests numpy pandas geopandas shapely python-dotenv folium
```

A Census API key is also required. Store it in a `.env` file at the 
repository root as `CENSUS_API_KEY=your_key_here`. 
Keys are free and available at api.census.gov/signup.

| Library | Purpose |
|---|---|
| requests | fetch remote data |
| numpy | numerical array operations |
| pandas | tabular data manipulation |
| geopandas | spatial data manipulation |
| shapely | geometric point objects |
| os, json, urllib.parse | file paths, JSON, URL encoding |
| io.StringIO | parse strings as files |
| dotenv | load environment variables |

---

In [7]:
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os, json, urllib.parse
from io import StringIO
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")
if not CENSUS_API_KEY:
    raise ValueError("CENSUS_API_KEY not found. Add it to .env at the repo root.")

MTA_URL_BASE   = "https://data.ny.gov/resource/wujg-7c2s.json"
MTA_CACHE      = os.path.join(DATA_DIR, "mta_agg.csv")

COVID_URL      = "https://raw.githubusercontent.com/nychealth/coronavirus-data/master/totals/data-by-modzcta.csv"
COVID_CACHE    = os.path.join(DATA_DIR, "covid_modzcta.csv")
COVID_RETRIEVED_DATE = "2026-05-10"

MODZCTA_URL    = "https://raw.githubusercontent.com/nychealth/coronavirus-data/master/Geography-resources/MODZCTA_2010_WGS1984.geo.json"
MODZCTA_CACHE  = os.path.join(DATA_DIR, "modzcta_clean.geojson")

ZCTA_URL       = "https://raw.githubusercontent.com/nychealth/coronavirus-data/master/Geography-resources/ZCTA-to-MODZCTA.csv"
ZCTA_CACHE     = os.path.join(DATA_DIR, "zcta_crosswalk.csv")

STATIONS_CACHE = os.path.join(DATA_DIR, "stations_with_modzcta.csv")
ACS_CACHE      = os.path.join(DATA_DIR, "acs_zcta.csv")
MERGED_CACHE   = os.path.join(DATA_DIR, "merged_modzcta.geojson")

## Step 2: MTA Subway Ridership

**Source:** MTA Subway Hourly Ridership, NY Open Data (Socrata API)
**Coverage:** January 2020 through December 2022

The source dataset contains more than 120 million hourly station-level
records. A SoQL (Socrata Query Language) query aggregates this to monthly
totals per station complex server-side before transmission, reducing the
result to roughly 17,500 rows.

The Socrata API is unreliable for queries of this scale, with response
times ranging from 6 minutes to over 40 minutes depending on server load,
and occasional timeouts regardless of client-side timeout settings.
For reproducibility, mta_agg.csv is committed directly to the repository.
The API query code is preserved below for reference but will only execute
if the cache file is missing.

In [8]:
if os.path.exists(MTA_CACHE):
    mta_agg = pd.read_csv(MTA_CACHE)
else:
    query = """
SELECT
  station_complex_id,
  station_complex,
  latitude,
  longitude,
  date_trunc_ym(transit_timestamp) AS month,
  SUM(ridership) AS monthly_ridership
WHERE
  transit_timestamp >= '2020-01-01T00:00:00'
  AND transit_timestamp <= '2022-12-31T23:59:59'
GROUP BY
  station_complex_id, station_complex, latitude, longitude, month
LIMIT 50000
"""
    url = MTA_URL_BASE + "?$query=" + urllib.parse.quote(query.strip())
    resp = requests.get(url, timeout=600)
    resp.raise_for_status()
    mta_agg = pd.read_json(StringIO(resp.text))
    mta_agg.to_csv(MTA_CACHE, index=False)

mta_agg["month"] = pd.to_datetime(mta_agg["month"])
mta_agg.head(3)

,station_complex_id,station_complex,latitude,longitude,month,monthly_ridership
0,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.91203,2020-01-01,438897.0
1,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.91203,2020-02-01,408850.0
2,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.91203,2020-03-01,228798.0


## Step 3: COVID-19 Outcomes by Neighborhood

**Source:** NYC Department of Health, coronavirus-data repository (GitHub)  
**Coverage:** Cumulative totals from February 29, 2020 through the most recent update

One row per Modified ZIP Code Tabulation Area (MODZCTA), the neighborhood 
geography used by NYC Health for disease surveillance. 

It contains confirmed case counts, death counts, and rates per 100,000 population for each of the city's 177 residential neighborhoods.

Data is downloaded directly from the NYC Health GitHub repository and cached locally.

Cumulative as of the retrieval date stored in COVID_RETRIEVED_DATE.
Cached values reflect the source file as it appeared on that date.

In [9]:
if os.path.exists(COVID_CACHE):
    covid = pd.read_csv(COVID_CACHE)
else:
    resp = requests.get(COVID_URL)
    resp.raise_for_status()
    covid = pd.read_csv(StringIO(resp.text))
    covid.to_csv(COVID_CACHE, index=False)
    
covid.head(3)

,MODIFIED_ZCTA,NEIGHBORHOOD_NAME,BOROUGH_GROUP,label,lat,lon,COVID_CONFIRMED_CASE_COUNT,COVID_PROBABLE_CASE_COUNT,COVID_CASE_COUNT,COVID_CONFIRMED_CASE_RATE,COVID_CASE_RATE,POP_DENOMINATOR,COVID_DEATH_COUNT,COVID_DEATH_RATE
0,10001,Chelsea/NoMad/West Chelsea,Manhattan,"10001, 10118",40.750693,-73.997137,9785,2606,12391,35436.09,44873.65,27613.09,73,264.37
1,10002,Chinatown/Lower East Side,Manhattan,10002,40.715781,-73.986176,25623,5818,31441,34017.63,41741.73,75322.71,493,654.52
2,10003,East Village/Gramercy/Greenwich Village,Manhattan,10003,40.731825,-73.989164,17512,3937,21449,32442.96,39736.70,53977.81,113,209.35


## Step 4: MODZCTA Boundary File

**Source:** NYC Department of Health, coronavirus-data repository (GitHub)

Polygon boundary file for New York City's 177 residential MODZCTAs. Each feature represents one neighborhood as a geographic shape.

One feature was removed before saving: MODZCTA 99999, a catch-all code assigned to records that could not be matched to a real geography (It is not a real place).

The coordinate reference system is confirmed as WGS 84 (EPSG:4326) on load, which is required for the spatial join in Step 7.

In [10]:
if os.path.exists(MODZCTA_CACHE):
    modzcta_gdf = gpd.read_file(MODZCTA_CACHE)
else:
    resp = requests.get(MODZCTA_URL)
    resp.raise_for_status()
    geojson = resp.json()

    # remove the catch-all MODZCTA 99999 feature
    geojson["features"] = [
        f for f in geojson["features"]
        if f["properties"].get("MODZCTA") != "99999"
    ]

    with open(MODZCTA_CACHE, "w") as fh:
        json.dump(geojson, fh)

    modzcta_gdf = gpd.read_file(MODZCTA_CACHE)

if modzcta_gdf.crs is None or modzcta_gdf.crs.to_epsg() != 4326:
    modzcta_gdf = modzcta_gdf.set_crs(epsg=4326)
    
modzcta_gdf.head(3)

,MODZCTA,label,geometry
0,10001,"10001, 10118","POLYGON ((-73.98774 40.74407, -73.98504 40.747..."
1,10002,10002,"POLYGON ((-73.9975 40.71407, -73.9973 40.71347..."
2,10003,10003,"POLYGON ((-73.98864 40.72293, -73.98843 40.723..."


## Step 5: ZCTA-to-MODZCTA Crosswalk

**Source:** NYC Department of Health, coronavirus-data repository (GitHub)

The Census Bureau reports data at the ZIP Code Tabulation Area (ZCTA) level. NYC Health reports at the MODZCTA level, which consolidates smaller ZCTAs to improve statistical reliability. This two-column lookup table maps 214 ZCTAs to their corresponding 177 MODZCTAs (the raw source has 215 rows; the catch-all MODZCTA 99999 is dropped on load).

This crosswalk is used in Step 8 to bridge Census income and poverty data to the neighborhood geography shared by the MTA and COVID datasets.

In [11]:
if os.path.exists(ZCTA_CACHE):
    zcta = pd.read_csv(ZCTA_CACHE)
else:
    resp = requests.get(ZCTA_URL)
    resp.raise_for_status()
    zcta = pd.read_csv(StringIO(resp.text))
    zcta = zcta[zcta["MODZCTA"].astype(str) != "99999"]
    
    zcta.to_csv(ZCTA_CACHE, index=False)

zcta.head(3)

,ZCTA,MODZCTA
0,10001,10001
1,10002,10002
2,10003,10003


## Step 6: American Community Survey -- Income and Poverty by ZIP Code

**Source:** U.S. Census Bureau, ACS 5-Year Estimates 2021 (Census API)  
**Variables:**
- B19013: Median household income
- B17001: Population below poverty level and total the population counted for poverty estimates

The Census API does not support state-level filtering for ZCTA geography, so the full national dataset is fetched and then filtered to ZIP codes appearing in the crosswalk from Step 5. This limits results to New York City.

Numeric columns are coerced on load. Any value the Census marks as unavailable or suppressed becomes NaN rather than causing a type error downstream.

---

In [12]:
if os.path.exists(ACS_CACHE):
    acs = pd.read_csv(ACS_CACHE)
else:
    # Census ACS5 does not support &in=state: for ZCTA geography. Fetch all, then filter
    acs_url = (
        "https://api.census.gov/data/2021/acs/acs5"
        "?get=NAME,B19013_001E,B17001_002E,B17001_001E"
        "&for=zip%20code%20tabulation%20area:*"
        f"&key={CENSUS_API_KEY}"
    )
    resp = requests.get(acs_url)
    resp.raise_for_status()
    rows = resp.json()
    acs = pd.DataFrame(rows[1:], columns=rows[0])

    acs = acs.rename(columns={
        "B19013_001E":              "median_income",
        "B17001_002E":              "poverty_count",
        "B17001_001E":              "poverty_universe",
        "zip code tabulation area": "zcta",
    })
    acs = acs[["NAME", "median_income", "poverty_count", "poverty_universe", "zcta"]]

    acs["median_income"]    = pd.to_numeric(acs["median_income"],    errors="coerce")
    acs["median_income"]    = acs["median_income"].where(acs["median_income"] > 0)

    acs["poverty_count"]    = pd.to_numeric(acs["poverty_count"],    errors="coerce")
    acs["poverty_universe"] = pd.to_numeric(acs["poverty_universe"], errors="coerce")
    acs["poverty_count"]    = acs["poverty_count"].where(acs["poverty_count"] >= 0)
    acs["poverty_universe"] = acs["poverty_universe"].where(acs["poverty_universe"] > 0)

    acs = acs[acs["zcta"].astype(str).isin(zcta["ZCTA"].astype(str))]
    acs.to_csv(ACS_CACHE, index=False)

acs.head(3)

,NAME,median_income,poverty_count,poverty_universe,zcta
2577,ZCTA5 10001,101409.0,3557,26023.0,10001
2578,ZCTA5 10002,37093.0,20554,76047.0,10002
2579,ZCTA5 10003,137533.0,4872,44965.0,10003


## Step 7: Spatial Join -- MTA Stations to MODZCTA

Each of the 428 MTA station complexes is represented as a geographic point 
using its latitude and longitude coordinates. A spatial join assigns each 
station to the MODZCTA polygon it falls within.

Five stations bordering Central Park failed the initial join. Their coordinates 
land inside the park boundary, which carries no MODZCTA designation. A 
distance-based fallback join assigns each of these stations to its nearest 
MODZCTA polygon.

The result is one row per station complex with its assigned MODZCTA attached.

---

In [13]:
if os.path.exists(STATIONS_CACHE):
    stations = pd.read_csv(STATIONS_CACHE)
else:
    # one row per station
    stations_dedup = (
        mta_agg
        .drop_duplicates(subset=["station_complex_id"])
        [["station_complex_id", "station_complex", "latitude", "longitude"]]
        .copy()
    )
    stations_dedup["latitude"]  = stations_dedup["latitude"].astype(float)
    stations_dedup["longitude"] = stations_dedup["longitude"].astype(float)

    # build GeoDataFrame
    geometry = [Point(lon, lat) for lon, lat in
                zip(stations_dedup["longitude"], stations_dedup["latitude"])]
    stations_gdf = gpd.GeoDataFrame(stations_dedup, geometry=geometry, crs="EPSG:4326")

    # spatial join
    joined = gpd.sjoin(stations_gdf, modzcta_gdf[["MODZCTA", "geometry"]],
                       how="left", predicate="within")

    stations = joined[["station_complex_id", "station_complex",
                        "latitude", "longitude", "MODZCTA"]].copy()

    mask = stations["MODZCTA"].isna()
    if mask.any():
        unmatched_ids = stations.loc[mask, "station_complex_id"]
        unmatched_gdf = stations_gdf.loc[
            stations_gdf["station_complex_id"].isin(unmatched_ids)
        ]

        # reproject to NY State Plane (feet) for accurate nearest-distance results
        stations_proj = unmatched_gdf.to_crs(epsg=2263)
        modzcta_proj  = modzcta_gdf.to_crs(epsg=2263)

        nearest = gpd.sjoin_nearest(
            stations_proj[["station_complex_id", "geometry"]],
            modzcta_proj[["MODZCTA", "geometry"]],
            how="left"
        )[["station_complex_id", "MODZCTA"]]

        stations = (
            stations
            .merge(nearest, on="station_complex_id", how="left", suffixes=("", "_nearest"))
        )
        stations["MODZCTA"] = stations["MODZCTA"].fillna(stations["MODZCTA_nearest"])
        stations.drop(columns=["MODZCTA_nearest"], inplace=True)
        stations.reset_index(drop=True, inplace=True)

    stations.to_csv(STATIONS_CACHE, index=False)

stations.head(3)

,station_complex_id,station_complex,latitude,longitude,MODZCTA
0,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.91203,11105
1,10,"49 St (N,R,W)",40.759900,-73.98414,10019
2,100,"Hewes St (M,J)",40.706870,-73.95343,11211


## Step 8: Aggregate and Merge

Four operations produce the final analysis-ready dataset.

**Ridership by neighborhood and year.** Monthly station ridership is joined to each station's MODZCTA from Step 7, then summed by neighborhood and year. The result is three annual totals per MODZCTA. Percent change from 2020 to 
2022 is computed as the rebound metric.

**Income and poverty by neighborhood.** Census data is aggregated from ZCTA to MODZCTA level using the crosswalk from Step 5. Poverty rate is computed correctly by summing numerator and denominator separately across all ZCTAs in a MODZCTA. Median income uses a population-weighted average. This is an approximation. True median aggregation requires microdata not available through the public API.

**Merge.** All three datasets are left-joined onto the 177-row MODZCTA geographic frame. Every join is left so no neighborhood is dropped, including the 54 MODZCTAs with no subway station.

**Income tier.** Neighborhoods are classified using fixed NYC-specific
income thresholds: below $50,000, $50,000 to $100,000, and above $100,000.
These represent broad material differences in household resources rather
than equal-sized statistical groups.

### Output: `merged_modzcta.geojson`
### Target: 177 features, 17 columns, geometry included.

---

In [14]:
if os.path.exists(MERGED_CACHE):
    merged_gdf = gpd.read_file(MERGED_CACHE)
else:
    # Step 1: MTA ridership pivot by MODZCTA and year
    mta_work = mta_agg.copy()
    mta_work["month"] = pd.to_datetime(mta_work["month"])
    mta_work["year"]  = mta_work["month"].dt.year

    stations_work = stations.copy()
    stations_work["MODZCTA"] = stations_work["MODZCTA"].astype(str)

    ridership = mta_work.merge(
        stations_work[["station_complex_id", "MODZCTA"]],
        on="station_complex_id", how="left"
    ).dropna(subset=["MODZCTA"])

    ridership_pivot = (
        ridership
        .groupby(["MODZCTA", "year"])["monthly_ridership"]
        .sum()
        .unstack("year")
    )
    ridership_pivot.columns = [f"ridership_{int(c)}" for c in ridership_pivot.columns]
    ridership_pivot["pct_change_20_22"] = (
        (ridership_pivot["ridership_2022"] - ridership_pivot["ridership_2020"])
        / ridership_pivot["ridership_2020"]
    )
    ridership_pivot = ridership_pivot.reset_index()

    # Step 2: ACS ZCTA to MODZCTA aggregation
    acs_work  = acs.copy()
    zcta_work = zcta.copy()
    acs_work["zcta"]     = acs_work["zcta"].astype(str)
    zcta_work["ZCTA"]    = zcta_work["ZCTA"].astype(str)
    zcta_work["MODZCTA"] = zcta_work["MODZCTA"].astype(str)

    acs_modzcta = acs_work.merge(zcta_work, left_on="zcta", right_on="ZCTA", how="left")

    def _wtd_income(grp):
        valid = grp[["median_income", "poverty_universe"]].dropna()
        valid = valid[valid["median_income"] > 0]
        if valid.empty:
            return np.nan
        return np.average(valid["median_income"], weights=valid["poverty_universe"])

    poverty_agg = acs_modzcta.groupby("MODZCTA").agg(
        _pov_count=("poverty_count",    "sum"),
        _pov_univ= ("poverty_universe", "sum"),
    )
    poverty_agg["poverty_rate"] = poverty_agg["_pov_count"] / poverty_agg["_pov_univ"]

    income_agg = (
        acs_modzcta.groupby("MODZCTA")
        .apply(_wtd_income, include_groups=False)
        .rename("median_income_wtd")
    )

    acs_agg = (
        poverty_agg[["poverty_rate"]]
        .join(income_agg)
        .reset_index()
    )
    acs_agg["MODZCTA"] = acs_agg["MODZCTA"].astype(str)

    # Step 3: Merge everything onto modzcta_gdf
    merged_gdf = modzcta_gdf.copy()
    merged_gdf["MODZCTA"] = merged_gdf["MODZCTA"].astype(str)

    merged_gdf = merged_gdf.merge(ridership_pivot, on="MODZCTA", how="left")

    covid_cols = ["MODIFIED_ZCTA", "NEIGHBORHOOD_NAME", "BOROUGH_GROUP",
                  "COVID_CASE_COUNT", "COVID_CASE_RATE",
                  "COVID_DEATH_COUNT", "COVID_DEATH_RATE", "POP_DENOMINATOR"]
    covid_sub = covid[covid_cols].copy()
    covid_sub["MODIFIED_ZCTA"] = covid_sub["MODIFIED_ZCTA"].astype(str)

    merged_gdf = merged_gdf.merge(
        covid_sub, left_on="MODZCTA", right_on="MODIFIED_ZCTA", how="left"
    ).drop(columns=["MODIFIED_ZCTA"])

    merged_gdf = merged_gdf.merge(acs_agg, on="MODZCTA", how="left")

    # Step 4: Income tier using NYC-specific dollar thresholds
    tier = pd.cut(
        merged_gdf["median_income_wtd"],
        bins=[0, 50000, 100000, float("inf")],
        labels=["working poor", "working class", "upper-middle class"]
    )
    # Convert Categorical to str; GeoJSON (pyogrio) cannot serialize Categorical dtype
    merged_gdf["income_tier"] = tier.astype(str).where(tier.notna(), other=None)

    merged_gdf.to_file(MERGED_CACHE, driver="GeoJSON")

merged_gdf.head(3)

,MODZCTA,label,geometry,ridership_2020,ridership_2021,ridership_2022,pct_change_20_22,NEIGHBORHOOD_NAME,BOROUGH_GROUP,COVID_CASE_COUNT,COVID_CASE_RATE,COVID_DEATH_COUNT,COVID_DEATH_RATE,POP_DENOMINATOR,poverty_rate,median_income_wtd,income_tier
0,10001,"10001, 10118","POLYGON ((-73.98774 40.74407, -73.98504 40.747...",33755218.0,39799422.0,57872904.0,0.714488,Chelsea/NoMad/West Chelsea,Manhattan,12391,44873.65,73,264.37,27613.09,0.136687,101409.0,upper-middle class
1,10002,10002,"POLYGON ((-73.9975 40.71407, -73.9973 40.71347...",8334043.0,10649137.0,14426530.0,0.731036,Chinatown/Lower East Side,Manhattan,31441,41741.73,493,654.52,75322.71,0.270280,37093.0,working poor
2,10003,10003,"POLYGON ((-73.98864 40.72293, -73.98843 40.723...",16644707.0,21078761.0,29453951.0,0.769569,East Village/Gramercy/Greenwich Village,Manhattan,21449,39736.70,113,209.35,53977.81,0.108351,137533.0,upper-middle class


## Step 9: Validation

Confirms all seven output files exist, meet minimum counts, and are readable. Any missing or undersized file prints FAIL.

Mirrored at the top of Part 2 to prevent the analysis notebook from running on incomplete data.

| File | Minimum | Failure likely means |
|---|---|---|
| mta_agg.csv | 17,000 rows | Socrata dataset restructured or retired |
| covid_modzcta.csv | 177 rows | NYC Health changed MODZCTA geography |
| modzcta_clean.geojson | 177 features | Boundary file updated upstream |
| zcta_crosswalk.csv | 214 rows | Crosswalk updated; re-examine 99999 filter |
| stations_with_modzcta.csv | 400 rows | Spatial join failed |
| acs_zcta.csv | 200 rows | Census API returned fewer ZCTAs than expected |
| merged_modzcta.geojson | 170 features | Merge dropped neighborhoods |

All geographic files use 2010 MODZCTA boundaries and 2021 ACS estimates (These are fixed by design). The MTA Socrata source is the most likely point of future failure.

---

In [15]:
checks = [
    (MTA_CACHE,      "mta_agg.csv",            17000),
    (COVID_CACHE,    "covid_modzcta.csv",         177),
    (MODZCTA_CACHE,  "modzcta_clean.geojson",     177),
    (ZCTA_CACHE,     "zcta_crosswalk.csv",        214),
    (STATIONS_CACHE, "stations_with_modzcta.csv", 400),
    (ACS_CACHE,      "acs_zcta.csv",              200),
    (MERGED_CACHE,   "merged_modzcta.geojson",    170),
]

passed = 0
for path, label, min_count in checks:
    if not os.path.exists(path):
        print(f"FAIL  {label}: file not found")
        continue

    size_kb = os.path.getsize(path) / 1024

    if path.endswith(".geojson"):
        df    = gpd.read_file(path)
        count = len(df)
        unit  = "features"
    else:
        df    = pd.read_csv(path)
        count = len(df)
        unit  = "rows"

    status = "PASS" if count >= min_count else "FAIL"
    if status == "PASS":
        passed += 1

    print(f"{status}  {label}: {count} {unit}, {size_kb:.1f} KB")

print(f"\n{passed}/{len(checks)} files passed.")

PASS  mta_agg.csv: 17539 rows, 1315.0 KB
PASS  covid_modzcta.csv: 177 rows, 21.2 KB
PASS  modzcta_clean.geojson: 177 features, 579.6 KB
PASS  zcta_crosswalk.csv: 214 rows, 2.5 KB
PASS  stations_with_modzcta.csv: 428 rows, 20.4 KB
PASS  acs_zcta.csv: 212 rows, 7.7 KB
PASS  merged_modzcta.geojson: 177 features, 702.1 KB

7/7 files passed.


## Data Notes and Limitations

**Five stations assigned by proximity.**

Five station complexes bordering Central Park fell outside all residential MODZCTA polygons during the spatial join. Each was assigned to its nearest MODZCTA by distance:

| Station | Assigned MODZCTA | Neighborhood |
|---|---|---|
| 96 St (B, C) | 10025 | Manhattan Valley / Morningside Heights / Upper West Side |
| 81 St (B, C) | 10024 | Upper West Side |
| 72 St (B, C) | 10023 | Lincoln Square |
| 59 St -- Columbus Circle (A, B, C, D, 1) | 10023 | Lincoln Square |
| 5 Av / 59 St (N, R, W) | 10019 | Hell's Kitchen / Midtown Manhattan |

---

**54 MODZCTAs have no subway station.**

30 are in Queens, 11 in Staten Island, transit absence is not necessarily synonymous with poverty. 

18 of the 54 are upper-middle class. In southeastern Queens specifically, no subway typically means bus routes with 30 to 60 minute trips to the nearest subway connection, or infrequent LIRR service at a significantly higher per-trip cost.

---

**Median income is approximated.**

Where multiple ZCTAs map to one MODZCTA, a population-weighted average
is used with the population counted for poverty estimates as the weight.
This is close to but not identical to total population, and not the same
as household count. The result is a defensible approximation for tier
classification, not a true MODZCTA median.

---

**Income tier cutoffs are fixed at $50,000 and $100,000.**

These thresholds reflect real material difference in New York City's cost of living. 

Actual ranges: 
- working poor $21,846 to $49,679 (n=29)
- working class $50,164 to $99,924 (n=100)
- upper-middle class $100,528 to $250,001 (n=48)